# Ella2027 — LTX Kaggle Worker (Run All)
نسخة تشغيل واحدة: شغّل Run All واترك جلسة Kaggle مفتوحة. يتم تثبيت المكتبات، فحص الاتصال، تحميل نموذج LTX، ثم استقبال طلبات Ella2027 تلقائياً.

**مرة واحدة فقط قبل التشغيل:** فعّل GPU وInternet من إعدادات الدفتر، واحفظ Kaggle Secret باسم `ELLA_KAGGLE_WORKER_TOKEN` بنفس قيمة متغير Render. لا تضع المفتاح داخل الكود.

ملاحظة: أول تشغيل يحتاج تنزيل النموذج الكبير. جودة الإخراج محسّنة بإعداد توليد أعلى ثم تكبير إخراج 2x وتحسين حدة خفيف؛ التكبير لا يعيد تفاصيل غير موجودة في المصدر. مدة الجلسة وحدود GPU في Kaggle ما زالت تنطبق.

In [ ]:
import os, sys, time, subprocess, shutil, gc
print('=== Ella2027 setup ===')
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg requests
import torch, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'فعّل GPU من Notebook Settings ثم أعد Run All.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
BASE_URL = 'https://ella2027.onrender.com'
TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
assert TOKEN, 'Kaggle Secret ELLA_KAGGLE_WORKER_TOKEN غير موجود.'
health = requests.get(BASE_URL + '/api/kaggle-worker/health', headers={'Authorization': 'Bearer ' + TOKEN}, timeout=60)
print('Render health:', health.status_code, health.text)
health.raise_for_status()
assert shutil.which('ffmpeg'), 'ffmpeg غير متوفر في جلسة Kaggle.'
print('Setup and connection OK.')

In [ ]:
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
MODEL_ID = 'Lightricks/LTX-Video'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Loading {MODEL_ID} with {dtype}; first run downloads model weights...')
pipe = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
    print('VAE tiling enabled.')
except Exception as e:
    print('VAE tiling not available; continuing:', str(e)[:180])
gc.collect()
torch.cuda.empty_cache()
print('LTX pipeline ready.')

In [ ]:
!mkdir -p /kaggle/working/ella_worker
!wget -q -O /kaggle/working/ella_worker/kaggle_worker.py https://raw.githubusercontent.com/WaelNameh84/Ella2027/main/services/kaggle-ltx-worker/kaggle_worker.py
sys.path.insert(0, '/kaggle/working/ella_worker')
from kaggle_worker import run_kaggle_ltx_worker

!wget -q -O /kaggle/working/ella_worker/ltx_generation.py https://raw.githubusercontent.com/WaelNameh84/Ella2027/main/services/kaggle-ltx-worker/ltx_generation.py
import importlib
import ltx_generation
importlib.reload(ltx_generation)
generate_with_ltx = ltx_generation.make_ltx_generator(pipe)

print('Ella2027 worker ready. Waiting for video jobs...')
run_kaggle_ltx_worker(BASE_URL, TOKEN, generate_with_ltx, idle_seconds=10)
